In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
import os
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL", "")

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [4]:
with open("011_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# The evaluation returns a structured JSON array where each object represents one test case result
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 ARN Region Extractor\n\n```python\ndef extract_region_from_s3_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs typically have the format: arn:aws:s3:::bucket-name\n    Note: S3 bucket ARNs don't contain region information directly.\n    However, this function handles cases where region might be included.\n    \n    Args:\n        arn (str): The S3 bucket ARN\n        \n    Returns:\n        str: The AWS region, or 'us-east-1' as default if not specified\n        \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        'us-east-1'\n        >>> extract_region_from_s3_arn('arn:aws:s3:us-west-2:123456789012:bucket/my-bucket')\n        'us-west-2'\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return 'us-east-1'\n    \n    try:\n        # Split ARN by colons\n        parts = arn.split(':')\n        \n        # Standard S3 ARN format: arn:aws:s3:::

Each result contains three key pieces of information:

- output: The complete response from Claude
- test_case: The original test case that was processed
- score: The evaluation score (currently hardcoded)

As you can see in the output, Claude generates quite verbose responses since we haven't provided specific formatting instructions yet. This is exactly the kind of issue we'll address as we refine our prompts.

At this point, we've successfully built the core evaluation pipeline. We can take our dataset, process it through Claude, and collect structured results. The major missing piece is the grading system - that hardcoded score of 10 needs to be replaced with actual evaluation logic.